# S6E10 TabR with route features

TabR-S (a retrieval-augmented MLP: each row attends to its nearest training rows) on the TabM v2 / RealMLP v6 feature set: original-data statistics, route (Flight Distance) passenger profile and in-fold target encoding.

In [ ]:
!pip install -q pytabkit skorch faiss-gpu-cu12  # TabR in pytabkit needs skorch and a GPU build of faiss
import faiss
assert hasattr(faiss, "GpuIndexFlatL2") and faiss.get_num_gpus() > 0, "faiss has no GPU support"
import glob, os, time, warnings
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from pytabkit import TabR_S_D_Classifier
warnings.filterwarnings("ignore")

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, name
    return hits[0]

train = pd.read_csv(find("train.csv")); test = pd.read_csv(find("test.csv"))
orig = pd.read_csv(find("data.csv"))
TARGET = "satisfaction"
y = train.pop(TARGET).astype(int).to_numpy()
orig[TARGET] = orig[TARGET].astype(int)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(train.shape, test.shape, orig.shape, DEVICE)


In [ ]:
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
RATINGS = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']
RATING_KEY = RATINGS
SERVICE_KEY = ["Customer Type", "Type of Travel", "Class"] + RATINGS

def original_stats(df: pd.DataFrame, orig: pd.DataFrame, smoothing: float = 20.0) -> pd.DataFrame:
    """Target statistics from the original dataset, looked up for each competition row.

    The original rows are disjoint from the competition data, so these features cannot leak the
    competition labels and are safe to compute once, outside the CV loop.
    """
    prior = orig[TARGET].mean()
    out = pd.DataFrame(index=df.index)
    raw_cols = CATEGORICAL + RATINGS + ["Age", "Flight Distance", "Departure Delay in Minutes"]
    for col in raw_cols:
        g = orig.groupby(col)[TARGET].agg(["sum", "count"])
        te = (g["sum"] + smoothing * prior) / (g["count"] + smoothing)
        out[f"orig_te_{col}"] = df[col].map(te).astype(float).fillna(prior)
    for name, key in [("ratings", RATING_KEY), ("service", SERVICE_KEY), ("full", [c for c in orig.columns if c != TARGET])]:
        g = orig.groupby(key, dropna=False)[TARGET].agg(["mean", "count"]).reset_index()
        g = g.rename(columns={"mean": f"orig_key_{name}_mean", "count": f"orig_key_{name}_n"})
        merged = df[key].merge(g, on=key, how="left")
        out[f"orig_key_{name}_mean"] = merged[f"orig_key_{name}_mean"].to_numpy()
        out[f"orig_key_{name}_n"] = merged[f"orig_key_{name}_n"].fillna(0).to_numpy()
    return out

O_tr, O_te = original_stats(train, orig), original_stats(test, orig)
O_tr, O_te = O_tr.drop(columns=["orig_key_full_mean", "orig_key_full_n"]), O_te.drop(columns=["orig_key_full_mean", "orig_key_full_n"])
O_med = O_tr.median()
O_tr, O_te = O_tr.fillna(O_med).astype("float32"), O_te.fillna(O_med).astype("float32")
print(O_tr.shape)


In [ ]:
X, X_test = train.drop(columns=["id"]), test.drop(columns=["id"])
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
COMBOS = [("Class", "Type of Travel", "Gender"), ("Class", "Type of Travel", "Age"),
          ("Food and drink", "Cleanliness"), ("Flight Distance", "Departure/Arrival time convenient"), ("Flight Distance",)]
combo_names = ["_".join(c) + "_" for c in COMBOS]

def engineer(df, ref):
    """ref is the training frame: category codes and counts are learned from it only."""
    out = df.copy()
    for c in cat_cols:
        out[c] = out[c].fillna("missing")
    for c in num_cols:
        out[c] = out[c].fillna(0.0)
    for c in num_cols:
        uniques = pd.Index(ref[c].fillna(0.0).unique())
        out[f"{c}_cat_"] = pd.Categorical(uniques.get_indexer(out[c]))
    for c in cat_cols + ["Leg room service"]:
        out[f"_{c}_count"] = out[c].map(ref[c].fillna("missing").value_counts()).fillna(0).astype("int32")
    for cols, name in zip(COMBOS, combo_names):
        key = lambda d: d[list(cols)].fillna(0).astype(str).agg("_".join, axis=1)
        uniques = pd.Index(key(ref).unique())
        out[name] = pd.Categorical(uniques.get_indexer(key(out)))
    return out

Xe, Xe_test = engineer(X, X), engineer(X_test, X)
def route_profile(train, test):
    """Label-free statistics of the passengers on each route (Flight Distance), pooled over train and test."""
    both = pd.concat([train, test], ignore_index=True)
    g = both.groupby("Flight Distance")
    prof = pd.DataFrame({
        "route_n": g.size(),
        "route_business": g["Class"].apply(lambda s: (s == "Business").mean()),
        "route_business_travel": g["Type of Travel"].apply(lambda s: (s == "Business travel").mean()),
        "route_loyal": g["Customer Type"].apply(lambda s: (s == "Loyal Customer").mean()),
        "route_age": g["Age"].mean(),
        "route_delay": g["Departure Delay in Minutes"].mean(),
        **{f"route_{c}": g[c].mean() for c in RATINGS[:6]},
    })
    f = lambda d: d[["Flight Distance"]].join(prof, on="Flight Distance").drop(columns="Flight Distance").astype("float32")
    return f(train), f(test)

R_tr, R_te = route_profile(train, test)
Xe, Xe_test = pd.concat([Xe, O_tr, R_tr], axis=1), pd.concat([Xe_test, O_te, R_te], axis=1)
print(Xe.shape, Xe_test.shape)


In [ ]:
params = dict(random_state=42, verbosity=0, val_metric_name="cross_entropy"  # TabR early-stops on accuracy or cross-entropy only, device=DEVICE,
              n_epochs=60, patience=8, batch_size=1024)


In [ ]:
oof, test_pred = np.zeros(len(y)), np.zeros(len(Xe_test))
drop_after_te = ["Class_Type of Travel_Age_", "Flight Distance_Departure/Arrival time convenient_", "Flight Distance_"]
for k, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(Xe, y)):
    t = time.time()
    X_tr, X_va, X_te = Xe.iloc[tr].copy(), Xe.iloc[va].copy(), Xe_test.copy()
    enc = TargetEncoder(cv=5, smooth="auto", shuffle=True, random_state=42)  # training fold only
    names = [f"_{c}TE" for c in combo_names]
    X_tr[names] = enc.fit_transform(X_tr[combo_names].astype(str), y[tr])
    X_va[names] = enc.transform(X_va[combo_names].astype(str))
    X_te[names] = enc.transform(X_te[combo_names].astype(str))
    X_tr, X_va, X_te = (d.drop(columns=drop_after_te) for d in (X_tr, X_va, X_te))
    model = TabR_S_D_Classifier(**params)
    model.fit(X_tr, y[tr], X_va, y[va])
    oof[va] = model.predict_proba(X_va)[:, 1]
    test_pred += model.predict_proba(X_te)[:, 1] / 5
    print(f"fold {k}: AUC {roc_auc_score(y[va], oof[va]):.6f}  ({time.time()-t:.0f}s)", flush=True)
    torch.cuda.empty_cache()
print(f"OOF AUC {roc_auc_score(y, oof):.6f}")
np.save("tabr_v1_oof.npy", oof); np.save("tabr_v1_test.npy", test_pred)
pd.DataFrame({"id": test["id"], TARGET: test_pred}).to_csv("submission.csv", index=False, float_format="%.6f")
